# CP201A Tract Codebook: Checking Census Tracts Across 2010 and 2020 Boundaries

**Fall 2026**

This notebook takes a list of census tracts for a neighborhood and reports how each tract
changed between the 2010 tract boundaries (used by the 2015 to 2019 ACS) and the 2020
boundaries (used by the 2020 to 2024 ACS). It then confirms with the Census API that the
tracts exist in both years, and checks how many residents live in group quarters.

It comes loaded with starter lists for the five field-trip neighborhoods. Section 3 is where
you put your own list.

Two files travel with this notebook:

* `tab20_tract20_tract10_st06.txt`: the Census Bureau's 2020 Tract to 2010 Tract
  Relationship File for California. Download page:
  https://www.census.gov/geographies/reference-files/time-series/geo/relationship-files.2020.html
* `CP201A_Case_Study_Neighborhoods_Starter_Tracts_2010_2020_Fall2026.xlsx`: the same
  results as a spreadsheet, with sources.

**What the relationship file is.** Every row is one piece of land that belongs to one 2010
tract and one 2020 tract, with its land area. A tract that did not change has one row with
itself. A tract that was split into two has two rows, one per piece. A tract whose boundary
was nudged has a big row with itself and one or more small rows with its neighbors. The
shares below are shares of **land area**, not population.

## 0. Setup

In [ ]:
%pip install -q census

In [ ]:
from census import Census
import pandas as pd
import numpy as np
import os

with open(os.path.expanduser('~/census_key.txt')) as f:
    api_key = f.read().strip()
c = Census(key=api_key)

In [ ]:
# The relationship file is published one state at a time. This is California (state FIPS 06).
# For another state, download its file from the page in the introduction and change the name here,
# for example 'tab20_tract20_tract10_st41.txt' for Oregon. The columns are the same in every state's file.
RELATIONSHIP_FILE = 'tab20_tract20_tract10_st06.txt'

cw = pd.read_csv(RELATIONSHIP_FILE, sep='|', dtype=str, encoding='utf-8-sig')
for col in ['AREALAND_TRACT_20', 'AREALAND_TRACT_10', 'AREALAND_PART']:
    cw[col] = pd.to_numeric(cw[col])
cw = cw[cw['AREALAND_PART'] > 0].copy()          # drop water-only pieces
cw['share_of_2020'] = cw['AREALAND_PART'] / cw['AREALAND_TRACT_20']
cw['share_of_2010'] = cw['AREALAND_PART'] / cw['AREALAND_TRACT_10']
print(f'{len(cw):,} tract pieces in this state')
cw.head()

## 1. The check

`check_tracts()` takes a list of six-digit tract codes and which numbering the list uses
(`'2010'` or `'2020'`), and returns one row per pair of overlapping tracts with a plain-English
label:

* **unchanged**: same number, same area (within 5 percent)
* **split from 2010 tract**: a 2020 tract that is a piece of a 2010 tract
* **renumbered**: same area, different number (UC Berkeley's campus tract 4226 became 9821)
* **boundary adjusted**: same number, but more than 5 percent of the area moved
* **absorbed 2010 tract (merge)** and **partial overlap**: rarer; look at the map

In [ ]:
def classify(r, vintage='2020', sliver=0.05, stable=0.95):
    same = r['GEOID_TRACT_20'] == r['GEOID_TRACT_10']
    s20, s10 = r['share_of_2020'], r['share_of_2010']
    anchor = s10 if vintage == '2010' else s20
    if anchor < sliver:
        return 'sliver (ignore)'
    if same and s20 >= stable and s10 >= stable:
        return 'unchanged'
    if same:
        return 'boundary adjusted'
    if s20 >= stable and s10 < stable:
        return 'split from 2010 tract' if r['GEOID_TRACT_20'][:9] == r['GEOID_TRACT_10'][:9] else 'part of 2010 tract (renumbered)'
    if s20 >= stable and s10 >= stable:
        return 'renumbered'
    if s20 < stable and s10 >= stable:
        return 'absorbed 2010 tract (merge)'
    return 'partial overlap'


def check_tracts(cw, state, county, tracts, vintage):
    '''Report how a list of tracts maps between 2010 and 2020 boundaries.

    Inputs:
    - cw: the relationship file DataFrame from Section 0
    - state, county: FIPS codes as strings ('06', '001')
    - tracts: list of six-digit tract codes
    - vintage: '2010' or '2020', whichever numbering the list uses

    Output: a DataFrame with one row per overlapping pair, slivers removed.
    '''
    ids = [f'{state}{county}{t}' for t in tracts]
    col = 'GEOID_TRACT_10' if vintage == '2010' else 'GEOID_TRACT_20'
    sub = cw[cw[col].isin(ids)].copy()
    not_found = sorted(set(ids) - set(sub[col]))
    if not_found:
        print('Not in the relationship file (check the number):', [t[-6:] for t in not_found])
    sub['change'] = sub.apply(classify, axis=1, vintage=vintage)
    sub = sub[sub['change'] != 'sliver (ignore)']
    sub['tract_2020'] = sub['GEOID_TRACT_20'].str[-6:]
    sub['tract_2010'] = sub['GEOID_TRACT_10'].str[-6:]
    out = sub[['tract_2020', 'NAMELSAD_TRACT_20', 'tract_2010', 'NAMELSAD_TRACT_10',
               'share_of_2020', 'share_of_2010', 'change']]
    return out.sort_values(['tract_2020', 'share_of_2020'], ascending=[True, False]).reset_index(drop=True)

## 2. The five field-trip neighborhoods

Starter lists. West Oakland and Bayview Hunters Point come from plans that published
their tract lists on 2010 boundaries; Chinatown, Berkeley, and Richmond are given on 2020
boundaries. See the spreadsheet's Read me sheet for sources.

In [ ]:
NEIGHBORHOODS = {
    'West Oakland': dict(state='06', county='001', vintage='2010', place='53000',
        tracts=['401400', '401500', '401600', '401700', '401800', '402200', '402400',
                '402500', '402600', '402700', '410500', '981900', '982000']),
    'San Francisco Chinatown': dict(state='06', county='075', vintage='2020', place='67000',
        tracts=['061101', '061102', '011800', '011300', '010701', '010702']),
    'Bayview Hunters Point': dict(state='06', county='075', vintage='2010', place='67000',
        tracts=['023200', '023103', '023001', '023400', '023102', '980600', '023300',
                '061200', '023003', '980900']),
    'Berkeley: campus, Southside, downtown (starter)': dict(state='06', county='001', vintage='2020', place='06000',
        tracts=['422300', '423000', '422400', '422500', '422700', '422800', '422901', '422902', '982100']),
    'Richmond: downtown and central, incl. Iron Triangle (starter)': dict(state='06', county='013', vintage='2020', place='60620',
        tracts=['376000', '377000', '378000', '379000', '380001', '380002']),
}

for name, n in NEIGHBORHOODS.items():
    print(f'\n=== {name} ({n["vintage"]} numbering, {len(n["tracts"])} tracts) ===')
    result = check_tracts(cw, n['state'], n['county'], n['tracts'], n['vintage'])
    changed = result[result['change'] != 'unchanged']
    print(f'{result["tract_2020"].nunique()} tracts in 2020; {result["tract_2010"].nunique()} in 2010; '
          f'{len(changed)} rows with a change')
    if len(changed):
        print(changed.round(3).to_string(index=False))

## 3. Your neighborhood

Put your own list here, say which numbering it uses, and run.

In [ ]:
MY_NAME = 'West Oakland'
MY_STATE, MY_COUNTY, MY_PLACE = '06', '001', '53000'
MY_VINTAGE = '2010'          # '2010' or '2020': the numbering your list uses
MY_TRACTS = NEIGHBORHOODS['West Oakland']['tracts']

my_check = check_tracts(cw, MY_STATE, MY_COUNTY, MY_TRACTS, MY_VINTAGE)
my_check.round(3)

In [ ]:
# The two lists you will actually pull: 2020 tracts for year=2024, 2010 tracts for year=2019
tracts_2020 = sorted(my_check['tract_2020'].unique())
tracts_2010 = sorted(my_check['tract_2010'].unique())
print('2020 tracts (year=2024):', tracts_2020)
print('2010 tracts (year=2019):', tracts_2010)

## 4. Confirm with the Census API, and check group quarters

Pull total population (B01003) and the group quarters population (B26001) for both lists in
their own years. This confirms the tracts really exist in the ACS for that year, shows which
98xx and 99xx tracts have almost nobody in them, and flags tracts where a large share of
residents live in group quarters (a campus, a jail, a nursing home), which describe the
institution rather than the neighborhood.

In [ ]:
check_vars = {
    'NAME': 'NAME',
    'B01003_001E': 'total_pop',
    'B01003_001M': 'total_pop_moe',
    'B26001_001E': 'group_quarters',
    'B26001_001M': 'group_quarters_moe',
}

def pull_check(state, county, tracts, year):
    df = pd.DataFrame(
        c.acs5.get(list(check_vars.keys()),
                   {'for': 'tract:*', 'in': f'state:{state} county:{county}'},
                   year=year)
    ).rename(columns=check_vars)
    df = df[df['tract'].isin(tracts)].copy()
    for col in ['total_pop', 'total_pop_moe', 'group_quarters', 'group_quarters_moe']:
        df[col] = pd.to_numeric(df[col])
    df = df.replace({-666666666: np.nan, -222222222: np.nan})
    df['gq_share'] = (df['group_quarters'] / df['total_pop'] * 100).round(1)
    missing = sorted(set(tracts) - set(df['tract']))
    if missing:
        print(f'Not returned for year={year}:', missing)
    return df[['NAME', 'tract', 'total_pop', 'total_pop_moe', 'group_quarters', 'gq_share']].sort_values('tract')

print('2020 to 2024 (2020 tracts):')
print(pull_check(MY_STATE, MY_COUNTY, tracts_2020, 2024).to_string(index=False))
print()
print('2015 to 2019 (2010 tracts):')
print(pull_check(MY_STATE, MY_COUNTY, tracts_2010, 2019).to_string(index=False))

Read the two tables together. A tract with a few hundred people and a large margin of error
adds noise and little information; a 98xx tract with zero residents can be dropped with one
sentence in your Data Notes; a tract where most residents are in group quarters is a decision
to make and to state. Whatever you decide goes in the introduction of your memo, with the
tract list and the boundary you cited.

In [ ]:
my_check.to_csv(f'tract_check_{MY_NAME.lower().replace(" ", "_")}.csv', index=False)
print('Saved.')